# Figure 4: public-sequence benchmarking and PREDICT molecular profiles

Figure 4 compares FastaLake databases selected from public sequences with specimen-matched metagenome databases in two cohorts (PREDICT stool, CapScan stool and intestinal fluid), audits why FastaLake accepted spectra the metagenome search did not, and shows PREDICT host and microbial profiles off and on antibiotics, across the Enterotype Dysbiosis Score, and across DNA, RNA, protein and metabolite layers. This notebook redraws every quantitative panel from `source_data/Fig4/`, recomputes every statistic and printed number, and compares each panel pixel by pixel with the panel as the original producer rendered it.

Run from the repository root: `python paper.py build Fig4`. Panels e, g and h and the paired antibiotic statistics need controlled clinical inputs (`FASTALAKE_CONTROLLED_DIR`, see `controlled/CONTROLLED_INPUTS.md`); without them those cells print SKIPPED. Pixel checks need the Söhne fonts (`FASTALAKE_FONT_DIR`).

In [ ]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()

import pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 200)
print("Söhne fonts present:", panels.fonts_present())

## Panel a

PREDICT study design and the matched molecular measurements (schematic). Supports "The PREDICT cohort comprised 306 stool LC–MS/MS acquisitions ... with companion DNA, RNA and metabolite measurements (Fig. 4a)" (v28d line 121). n: 306 acquisitions, 296 with a matched metagenome. No test. Shipped as artwork (`artwork/panel_a.pdf`); every typed number is traced in `ARTWORK_NUMBERS.md`. Shown here from the published composite raster.

In [ ]:
# The published composite shows the controlled panels e, g, h and is held with the controlled
# inputs; the public repository ships panel a as artwork/panel_a.png (crop of that composite).
_comp = panels.held_file(UNIT, "expected", "Fig4_v30.png")
display(Image(filename=str(panels.artwork_preview(_comp, OUT / "panel_a.png") if _comp else UNIT / "artwork" / "panel_a.png")))
roster = panels.load_table("predict_acquisition_roster.tsv")
print("PREDICT acquisitions:", len(roster), "| with matched metagenome:", int(roster.matched_metagenome.sum()))

## Panel b

Peptides of 9–50 residues per acquisition in the matched metagenome (MG) and FastaLake (FL) searches after 10-fold spectrum cross-fitting: each spectrum is searched against a FastaLake database selected from the predictions of the other nine folds. Supports the cross-fitted gains in v28d line 123 ("−3.5% in PREDICT (95% CI −4.8 to −1.4%), −5.5% in CapScan stool (95% CI −8.4 to −2.3%) and 28.3% in intestinal fluid (95% CI 20.6–35.3%), and 21–36% across the four intestinal sampling regions") and line 133. n: 296 PREDICT acquisitions (97 participant clusters), 56 CapScan stool (15 participants), 239 CapScan intestinal fluid (15 participants). Uncertainty: percentile participant-cluster bootstrap, 10,000 draws, seed 20261004 (Methods line 777). The printed interval depends on the random stream: the cell also prints the CapScan stool interval from a separate per-region stream and from the stool stratum alone under 21 seeds (a repository sensitivity check, not in the manuscript). The panel is drawn together with panel c on one page, as by the producer. The line 129 mixed model (fluid vs stool, 5–50 residues) uses the same table. Its printed P values are an unadjusted Wald z test (P = 3 × 10⁻⁷) and a 3-df Wald chi-square of the region contrasts (P = 0.61); Methods line 229 describes a likelihood-ratio omnibus test and Holm-adjusted Wald contrasts, so the cell also prints the likelihood-ratio statistic (OPEN_ITEMS MISMATCH).

In [ ]:
x = panels.load_crossfit()
panels.panel_bc(x, OUT / "panel_bc.pdf")
display(Image(filename=str(panels.render_png(OUT / "panel_bc.pdf", OUT / "panel_bc.png", clip=panels.BC_CROP_PT))))
boot = panels.stat_bc_crossfit_bootstrap(x)
print(boot[boot.min_length == panels.L_BC].to_string(index=False))
region = panels.stat_bc_region_bootstrap(x)
print(region[region.min_length == panels.L_BC].to_string(index=False))
sens = panels.stat_bc_bootstrap_seed_sensitivity(x)
print(sens.to_string(index=False))
print("stool 9-50 CI, stool stratum alone, seed", panels.BOOT_SEED_CROSSFIT, "and", len(sens) - 1, "further seeds: upper bound",
      sens.ci95_high.min(), "to", sens.ci95_high.max(), "| lower bound", sens.ci95_low.min(), "to", sens.ci95_low.max())
print("same-spectrum gain > cross-fitted gain (9-50, v30):", panels.stat_bc_same_vs_crossfit(x))
print("same-spectrum gain > cross-fitted gain (5-50, v28d):", panels.stat_bc_same_vs_crossfit(x, L=panels.L_SAME))
mm = panels.stat_mixed_model(x)
print({k: mm[k] for k in ("n_acq", "n_part", "fit_method", "fold", "fold_ci_low", "fold_ci_high", "z", "p", "region_wald_chi2", "region_p", "lrt_chi2", "lrt_p")})
MM_KEYS = ("length", "n_acq", "n_part", "fit_method", "fold", "fold_ci_low", "fold_ci_high", "z", "p", "region_wald_chi2", "region_p", "lrt_chi2", "lrt_p")
mm_same = panels.stat_mixed_model(x, view="same_spectrum")
mm_5 = panels.stat_mixed_model(x, L=panels.L_SAME)
print("same-spectrum, 9-50:", {k: mm_same[k] for k in MM_KEYS})
print("cross-fitted, 5-50 (v28d):", {k: mm_5[k] for k in MM_KEYS})


## Panel c

Fraction of the metagenome-search peptide identities (9–50 residues) that the cross-fitted FastaLake search also identified in the same acquisition. Drawn in the Panel b cell. The v30 text (Results paragraph 65) prints the medians 78.4% (PREDICT), 72.0% (CapScan stool) and 75.4% (intestinal fluid) with "(Fig. 4b,c and Source Data)". n as in panel b; unit acquisition. No test.

In [ ]:
print(panels.stat_bc_recovery(x))

## Panel d

Metagenome-search outcome for the 6.69 million PREDICT spectra accepted only by FastaLake (peptide q ≤ 0.01 with FastaLake, not with the metagenome search). Supports v28d line 131 ("absent from the metagenome database in 29% of cases. In about 69%, the sequence was present ... but the highest-scoring reported metagenome target failed the peptide threshold (Fig. 4d)"). n: 6,689,188 spectra from 296 acquisitions; unit spectrum. No test; percentages of the total.

In [ ]:
audit_table = panels.load_table("spectrum_audit_per_acquisition.tsv")
panels.panel_d(audit_table, OUT / "panel_d.pdf")
display(Image(filename=str(panels.render_png(OUT / "panel_d.pdf", OUT / "panel_d.png"))))
audit = panels.stat_d_audit(audit_table)
print(audit)

## Panel e

Human-assigned share of LFQ intensity in 42 patients sampled off and on antibiotics (boxes: median and IQR; whiskers: 1.5 IQR). Supports v28d line 137: "the human-assigned share of total label-free quantification (LFQ) intensity was 11.1 percentage points higher during exposure (95% CI 5.6–17.2, Benjamini–Hochberg q=0.00068)". n: 42 patients, each off and on (repeated within patient). Test: two-sided Monte Carlo sign-flip test of the mean paired difference (200,000 flips); CI by percentile patient bootstrap (50,000 draws, seed 20260907); BH over the two antibiotic endpoints. Needs the controlled paired table. This repository is executed without controlled inputs, so the stored output of this cell shows no image.

In [ ]:
st = panels.load_table("clinical_paired_statistics.tsv").set_index("metric")
ab = panels.load_paired_controlled()
paired = None
if ab is not None:
    panels.panel_e(ab, st, OUT / "panel_e.pdf")
    display(Image(filename=str(panels.render_png(OUT / "panel_e.pdf", OUT / "panel_e.png"))))
    paired = panels.stat_eg_paired(ab)
    print(paired.to_string(index=False))
else:
    print("public aggregate statistics (clinical_paired_statistics.tsv):")
    print(st.to_string())

## Panel f

Mean origin composition of LFQ signal off and on antibiotics (42 patients; human-assigned, microbial species, microbial above species, other annotated origins, unresolved origin), with percentages printed for segments of at least 10%. Supports the description in the legend (v28d line 127, "conditional origin composition (f)"). Aggregate of 42 patients; no test.

In [ ]:
printed = panels.panel_f(panels.load_table("clinical_composition_means.tsv"), OUT / "panel_f.pdf")
display(Image(filename=str(panels.render_png(OUT / "panel_f.pdf", OUT / "panel_f.png"))))
print(pd.DataFrame(printed).to_string(index=False))

## Panel g

Quantified protein groups in the same 42 patients off and on antibiotics. Supports v28d line 137: "The number of quantified protein groups was lower by 2,214 (95% CI 998–3,469, q=0.0013, Fig. 4e–g)". n, test and interval as in panel e (sign-flip seed 20260908). Needs the controlled paired table. This repository is executed without controlled inputs, so the stored output of this cell shows no image.

In [ ]:
if ab is not None:
    box = panels.panel_g(ab, st, OUT / "panel_g.pdf")
    display(Image(filename=str(panels.render_png(OUT / "panel_g.pdf", OUT / "panel_g.png"))))
    print([{k: float(b[k]) for k in ("q1", "med", "q3", "whislo", "whishi")} for b in box])

## Panel h

Five host proteins, six microbial KEGG pathways and eight metabolites across 85 patients ordered by the metagenomic Enterotype Dysbiosis Score; values are within-feature z scores clipped to ±2 for display. Supports v28d line 137 ("we displayed host proteins, microbial pathways and metabolites across patients ordered by EDS (Fig. 4h)"). n: 85 patients. No test (display only; the associations are tested in Supplementary Fig. 10). The antibiotic track is a controlled input; without it the panel is not drawn. This repository is executed without controlled inputs, so the stored output of this cell shows no image.

The top track is labelled "Enterotype dysbiosis score (0–1)" as in v30 (`panels.H_EDS_TRACK_LABEL`; v28f printed "EDS (0–1)").

In [ ]:
Z, feat, tracks = panels.load_heatmap()
abx = panels.load_antibiotic_track()
if panels.panel_h(Z, feat, tracks, abx, OUT / "panel_h.pdf"):
    display(Image(filename=str(panels.render_png(OUT / "panel_h.pdf", OUT / "panel_h.png"))))
print("features shown:", int((feat.n_positive_85 > 0).sum()), "| patients:", len(tracks))

## Panel i

Correlation between patient-profile distances of the molecular layers (77 patients with DNA, RNA, protein and metabolite data) for native, eggNOG OG and KEGG KO features; protein excludes human-assigned signal. Supports v28d line 139: "distances between protein profiles excluding human-assigned signal correlated with DNA, RNA and metabolite distances (Spearman ρ=0.44, 0.45 and 0.44, respectively, all q<0.001, Fig. 4i)". n: 77 patients (2,926 patient pairs). Test: Mantel-type Spearman correlation of distance matrices with 1,999 patient-label permutations (seed 20260907), two-sided; BH within the native family (10 pairs) and the functional family (28 pairs over OG, KO and pathway level). The cells are drawn from the upstream correlation tables; the same cell recomputes every correlation from the distance matrices and compares.

In [ ]:
native_up = panels.load_table("crossomics_native_correlations.tsv")
functional_up = panels.load_table("crossomics_functional_correlations.tsv")
cells = panels.panel_i(native_up, functional_up, OUT / "panel_i.pdf")
cbar = panels.panel_i_colourbar(feat, tracks, OUT / "panel_i_colourbar.pdf")
display(Image(filename=str(panels.render_png(OUT / "panel_i.pdf", OUT / "panel_i.png"))))
dist = panels.load_table("crossomics_distances_predict77.tsv")
native = panels.stat_i_native(dist)
functional = panels.stat_i_functional(dist)
print(native.to_string(index=False))
up = native_up[(native_up.cohort == "PREDICT") & (native_up.analysis == "sensitivity_all_features")]
m = native.merge(up, on=["layer_a", "layer_b"], suffixes=("", "_upstream"))
print("native: max |rho - upstream|", (m.rho - m.rho_upstream).abs().max(), "| max |q - upstream|", (m.q - m.q_upstream).abs().max())
upf = functional_up[(functional_up.cohort == "PREDICT") & (functional_up.scope == "displayed77") & (functional_up.prevalence == 0)]
m = functional.merge(upf, on=["level", "layer_a", "layer_b"], suffixes=("", "_upstream"))
print("functional: max |rho - upstream|", (m.rho - m.rho_upstream).abs().max(), "| max |q - upstream|", (m.q - m.q_upstream).abs().max(), "| tests", len(m))
print("colour bar (pt):", cbar)

## Text numbers

Numbers in the Results paragraphs on Figure 4 (v28d lines 121–139) that no panel prints: same-spectrum benchmark values (line 123; Supplementary Fig. 6d,e in v28f), same-spectrum gains by CapScan sampling site, DNA amounts and database sizes (line 129). Units: acquisition, and participant for the database comparison.

The v30 text prints the same-spectrum values at 9–50 residues (`panels.stat_text_same_spectrum`): PREDICT and CapScan gains, acquisitions with more peptides and recoveries (Results paragraphs 64 and 65, cited to Supplementary Fig. 6d,e), and the fluid/stool yields (paragraph 70). The mixed model (paragraph 69) is fitted on 9–50-residue cross-fitted counts (`panels.MIXED_MODEL_LENGTH`), and the same-spectrum versus cross-fitted comparison (paragraph 70) is at 9–50 residues.

In [ ]:
bench = panels.load_table("same_spectrum_benchmark.tsv")
print(bench.groupby(["cohort", "min_length"]).agg(n=("acquisition_id", "size"), median_ratio=("count_ratio", "median"),
                                                  median_recovery=("mg_recovery_fraction", "median")).to_string())
text = panels.stat_text_capscan(x, panels.load_table("capscan_dna_yield_by_acquisition.tsv"))
for k, v in text.items():
    print(k, v)
ss = panels.stat_text_same_spectrum(bench)
for k, v in ss.items():
    print(k, v)


## Checks and numbers

Pixel checks of every quantitative panel against the original producer's output (300 dpi), a negative control (the regenerated panel-i colour-bar page against the expected b,c page), and every recomputed number in `NUMBERS.tsv` order.

Panels e, g and h are drawn from controlled inputs. Their producer renders and check receipts are not shipped in the repository (OPEN_ITEMS AUTHOR); with `FASTALAKE_CONTROLLED_DIR` set, the references are read from `$FASTALAKE_CONTROLLED_DIR/Fig4_expected/` and the receipts are written to `build/Fig4/checks_controlled/`.

The v30 checks compare the regenerated b,c page and (with controlled inputs) panel h with the same regions of the v30 composite `expected/Fig4_v30.png`, and the v30 composite with the v28f composite.

In [ ]:
for r in panels.run_pixel_checks(OUT, UNIT):
    print(r)
for r in panels.check_against_v30(OUT, UNIT):
    print(r)
rows = panels.collect_numbers(x, boot, region, mm, paired, native, functional, audit, text, roster, bench, len(tracks))
for key, value in rows:
    print(f"{key}\t{value!r}")